# PtX–UHS capacity expansion and operational analysis — PES-GM 2027 · v0.2

**Single-node 2050 PyPSA-Island model | Taranaki resource case | Tank, depleted gas reservoir (DGR) and salt cavern (SC).**

This notebook follows the organisation and **manuscript plotting conventions of the Punta Arenas notebook**: shared imports and execution switches, simulation and export, main-paper figures, then supplementary/sensitivity analysis. It calls the **original `config.py` unchanged** and reads only CSV files from `input_uhs_pesgm/`. There are **no auxiliary project Python modules** (`bridge_build.py`, `run_pilot.py`, `config_pilot.py` or `test_inputs.py`).

**Interpretation:** energy and power capacities are optimised endogenously. DGR costs, site-size linearisation and well deliverability are screening assumptions. SC is a technology comparator and is **not assumed geologically available in Taranaki**. No real Taranaki renewable profiles were supplied with the original materials; synthetic profiles are allowed **for debugging only**.

## 1. Imports, run controls and manuscript style

The plotting palette, figure width, font sizes, shared axis helpers, export DPI and directory structure follow the Punta Arenas analysis. Expensive model runs and sensitivity sweeps are **off by default**.

In [1]:
# Standard library and scientific stack
from pathlib import Path
import importlib
import importlib.util
import sys
import types
import warnings
import gc

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from IPython.display import display

# The user's unmodified config.py imports build_demand at import time, although
# this standalone pilot calls build_case_network() directly and does not use it.
# Supply an in-memory placeholder only when that optional file is not present.
if "build_demand" not in sys.modules and importlib.util.find_spec("build_demand") is None:
    _demand_placeholder = types.ModuleType("build_demand")
    def _build_demands_unavailable(*args, **kwargs):
        raise RuntimeError("build_demand.py is required by config.run_case(), but not by this pilot.")
    _demand_placeholder.build_demands = _build_demands_unavailable
    sys.modules["build_demand"] = _demand_placeholder

try:
    import config
    config = importlib.reload(config)
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        f"Cannot import config.py ({exc}). Keep the original config.py in the same directory "
        "and install PyPSA, Linopy and the selected solver in this notebook kernel."
    ) from exc
config.configure_runtime_output()

plt.style.use("seaborn-v0_8-whitegrid")

c:\Users\vse23\AppData\Local\anaconda3\envs\pypsa-isolated\Lib\site-packages\google\api_core\_python_version_support.py:275: FutureWarning:

You are using a Python version (3.11.14) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.



In [2]:
# ── Execution controls ─────────────────────────────────────────────────────
CASE_FOLDER = Path("input_uhs_pesgm")
OUTPUT_ROOT = Path("output") / "output_uhs_pesgm"
SIMULATION_EXPORT_DIR = OUTPUT_ROOT / "simulations"
TABLE_EXPORT_DIR = OUTPUT_ROOT / "tables"
PAPER_FIGURE_DIR = OUTPUT_ROOT / "figures"
for _folder in [SIMULATION_EXPORT_DIR, TABLE_EXPORT_DIR, PAPER_FIGURE_DIR]:
    _folder.mkdir(parents=True, exist_ok=True)

MODEL_YEAR = 2050
RUN_SIMULATIONS = False          # Set True after supplying real renewable profiles
RUN_SENSITIVITY = False          # Separate and computationally expensive sweep
OVERWRITE_EXISTING_EXPORTS = False
SAVE_NETWORK_NETCDF = False     # Optional; long hourly networks can be large
SOLVER = "highs"                # Or "gurobi" (if configured locally)

# Scenario IDs are listed in input_uhs_pesgm/scenarios_2050.csv.
RUN_SCENARIO_CODES = ["T-S2", "C-S2"]
ACTIVE_SCENARIO = "C-S2"

PROFILE_CSV = CASE_FOLDER / "Taranaki_CF_2050.csv"
DEMO_SYNTHETIC_CF = False        # NEVER use True for manuscript results

# ── Plotting conventions preserved from Punta Arenas ───────────────────────
PAPER_MODE = True
PAPER_DPI = 300
CM_TO_INCH = 1 / 2.54
WORD_DOUBLE_COL_IN = 16.0 * CM_TO_INCH
FIGSIZE_2COL = (WORD_DOUBLE_COL_IN, 3.35)
FIGSIZE_PAPER_2X2 = (WORD_DOUBLE_COL_IN, 4.95)
FIGSIZE_2COL_LARGE = (7.2, 5.6)
FONT_SIZE_BASE, FONT_SIZE_SMALL, FONT_SIZE_LABEL, FONT_SIZE_PANEL = 9, 8, 9, 9
plt.rcParams.update({
    "font.size": FONT_SIZE_BASE,
    "axes.labelsize": FONT_SIZE_BASE,
    "axes.titlesize": FONT_SIZE_BASE,
    "xtick.labelsize": FONT_SIZE_SMALL,
    "ytick.labelsize": FONT_SIZE_SMALL,
    "legend.fontsize": FONT_SIZE_SMALL,
})

# Exact corresponding technology colours used in the Punta Arenas notebook.
TECH_COLORS = {
    "wind": "#5bbcff", "solar": "#e0a106", "electrolyzer": "#00897b",
    "h2_fuel_cell": "#ff8f00", "hydrogen_storage": "#26a69a",
    "uhs_injection": "#1e88e5", "uhs_withdrawal": "#3949ab",
    "uhs_storage": "#8e24aa", "asu": "#80cbc4", "dac": "#4db6ac",
    "slack": "#d32f2f",
}
STORAGE_COLORS = {"Tank": TECH_COLORS["hydrogen_storage"],
                  "DGR": TECH_COLORS["uhs_storage"],
                  "SC": TECH_COLORS["uhs_injection"]}
PATHWAY_LABELS = {"S1": "PtH₂tP", "S2": "PtA", "S3": "PtM",
                  "S4": "PtH₂tP + PtA", "S5": "PtH₂tP + PtM"}

def _paper_panel_label(ax, label: str, x=0.015, y=0.965):
    ax.text(x, y, label, transform=ax.transAxes, ha="left", va="top",
            fontsize=FONT_SIZE_PANEL, fontweight="bold",
            bbox=dict(facecolor="white", edgecolor="none", alpha=.75, pad=1.5), zorder=20)

def _paper_axis(ax, xlabel=None, ylabel=None, grid_axis="y"):
    if xlabel is not None: ax.set_xlabel(xlabel)
    if ylabel is not None: ax.set_ylabel(ylabel)
    ax.grid(grid_axis is not None, axis=grid_axis or "both", alpha=.25)
    ax.set_axisbelow(True)
    ax.tick_params(axis="both", labelsize=FONT_SIZE_SMALL)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    return ax

def _save_paper_figure(fig, stem):
    fig.tight_layout()
    for extension in ("png", "svg"):
        fig.savefig(PAPER_FIGURE_DIR / f"{stem}.{extension}", dpi=PAPER_DPI,
                    bbox_inches="tight", facecolor="white")
    print("Saved:", PAPER_FIGURE_DIR / f"{stem}.png", "and .svg")

## 2. Inputs and scenario configuration

**CSV files are the only run-time source of assumptions.** All technology costs are already annualised in **EUR/MW/year or EUR/MWh/year** (product stores: EUR/t/year). The old Excel is a separate screening tool; editing it **does not** automatically overwrite these CSVs. The physical input columns and units are checked below.

The network builder uses `config.build_case_network()` and `config.add_custom_constraints()` from your existing `config.py`; there is no second model implementation.

In [4]:
# ── Data loading and validation ────────────────────────────────────────────
def _read_required_csv(name):
    path = CASE_FOLDER / name
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Keep the notebook beside input_uhs_pesgm/.")
    return pd.read_csv(path)

_assumption_table = _read_required_csv("assumptions_2050.csv")
if not {"parameter", "value"}.issubset(_assumption_table.columns):
    raise ValueError("assumptions_2050.csv requires 'parameter' and 'value' columns")
if _assumption_table["parameter"].duplicated().any():
    raise ValueError("Duplicated parameter names in assumptions_2050.csv")
p = _assumption_table.set_index("parameter")["value"].astype(float).to_dict()
assert int(p["year"]) == MODEL_YEAR

scenarios_df = _read_required_csv("scenarios_2050.csv")
expected_columns = {"case_id", "technology", "pathway", "dispatch",
                    "power_twh", "ammonia_kt", "methanol_kt", "max_fec"}
if not expected_columns.issubset(scenarios_df.columns):
    raise ValueError(f"Missing scenario columns: {sorted(expected_columns - set(scenarios_df.columns))}")
if scenarios_df["case_id"].duplicated().any():
    raise ValueError("Scenario IDs must be unique")
assert len(scenarios_df) == 21, "Expected the current 21 pilot scenarios"
for key in ["power_twh", "ammonia_kt", "methanol_kt", "max_fec"]:
    scenarios_df[key] = pd.to_numeric(scenarios_df[key], errors="raise")
if not scenarios_df["technology"].isin(["Tank", "DGR", "SC"]).all():
    raise ValueError("Unknown storage technology in scenarios_2050.csv")
if (scenarios_df[["power_twh", "ammonia_kt", "methanol_kt"]] < 0).any().any():
    raise ValueError("Demand entries cannot be negative")

snapshots = pd.date_range(f"{MODEL_YEAR}-01-01", periods=8760, freq="h")
print(f"Loaded {len(scenarios_df)} scenarios from {CASE_FOLDER}/")
display(scenarios_df[["case_id", "technology", "pathway", "dispatch", "power_twh", "ammonia_kt", "methanol_kt"]])

Loaded 21 scenarios from input_uhs_pesgm/


,case_id,technology,pathway,dispatch,power_twh,ammonia_kt,methanol_kt
0,T-S1-D,Tank,S1,daily,1.0,0.0,0.0
1,T-S1-S,Tank,S1,seasonal,1.0,0.0,0.0
2,T-S2,Tank,S2,none,0.0,636.0,0.0
3,T-S3,Tank,S3,none,0.0,0.0,360.0
4,T-S4-D,Tank,S4,daily,1.0,636.0,0.0
5,T-S4-S,Tank,S4,seasonal,1.0,636.0,0.0
6,T-S5-D,Tank,S5,daily,1.0,0.0,360.0
7,T-S5-S,Tank,S5,seasonal,1.0,0.0,360.0
8,D-S1-S,DGR,S1,seasonal,1.0,0.0,0.0
9,D-S2,DGR,S2,seasonal,0.0,636.0,0.0


## 3. Shared physical components and dispatch assumptions

The functions below replace `run_pilot.py` **inside this notebook**. PtH₂tP electricity is served only from the fuel cell, not directly from the renewable bus. Product shipments occur in 48-hour monthly loading windows, and synthesis auxiliaries use electricity explicitly. These demand patterns are **assumptions to test**, not observed operation. Tank and UHS injection/withdrawal rates are optimised independently of stored energy.

In [5]:
def make_hourly_profile(path: Path|None, *,demo:bool, n_hours:int=8760):
    """Require a real 8760-hour wind/solar file, except explicit synthetic smoke test."""
    if demo:
        t=np.arange(n_hours,dtype=float)
        wind=np.clip(0.43+0.16*np.cos(2*np.pi*t/(24*14))+0.13*np.cos(2*np.pi*t/(24*365)+0.3),0,1)
        solar=np.clip((0.28+0.12*np.cos(2*np.pi*t/(24*365)-0.2))*np.maximum(0,np.sin(2*np.pi*(t%24-6)/24)),0,1)
        print('WARNING: SYNTHETIC DEMO wind/solar CF — NOT VALID FOR PAPER RESULTS')
        return wind,solar
    if path is None:raise ValueError('Real 8760-hour Taranaki renewable profile CSV is required, or use --demo for code tests')
    df=pd.read_csv(path,comment='#')
    if not {'wind_cf','solar_cf'}.issubset(df.columns):
        raise ValueError('Profile must have columns wind_cf,solar_cf')
    if len(df)!=n_hours:raise ValueError(f'Expected {n_hours} hourly rows; received {len(df)}')
    for key in ('wind_cf','solar_cf'):
        values=pd.to_numeric(df[key],errors='coerce')
        if values.isna().any() or not values.between(0,1).all():
            raise ValueError(f'{key} must contain numeric 0–1 values; got invalid values')
    return df['wind_cf'].to_numpy(dtype=float),df['solar_cf'].to_numpy(dtype=float)

def electricity_service(p, case, snapshots):
    """A separately served power bus enforces PtH2->fuel-cell->electricity."""
    demand=np.zeros(len(snapshots), dtype=float)
    if case['power_twh']<=0:return demand
    target=case['power_twh']*1e6 # MWh electric per year
    if case['dispatch']=='daily':
        # Four hours per day, local-clock time assumed to match profile indices.
        h=int(round(p['daily_peak_hours']))
        if h<1 or h>24:raise ValueError('daily_peak_hours must be 1..24')
        selected=(snapshots.hour>=18)&(snapshots.hour<18+h) if h<=6 else (snapshots.hour<h)
        demand[selected]=target/selected.sum()
    elif case['dispatch']=='seasonal':
        # NZ winter stress-test: 90 days from June 1 (configurable below).
        day_start=pd.Timestamp('2050-06-01')
        days=int(round(p['seasonal_full_load_days_y']))
        selected=(snapshots>=day_start)&(snapshots<day_start+pd.Timedelta(days=days))
        if selected.sum()==0:raise RuntimeError('Seasonal load mask empty')
        demand[selected]=target/selected.sum()
    else:raise ValueError('Nonzero power requires daily or seasonal service')
    assert abs(demand.sum()-target)<1e-5
    return demand

def h2_assets(case,p):
    base=dict(zone='taranaki',available_from_year=2050,earliest_commissioning_year=2050,
              commissioning_type='flexible',enabled=True,installed_capacity_mw=0,
              installed_energy_mwh=0,standing_loss=0,initial_inventory_fraction=np.nan,
              final_inventory_fraction=np.nan,cyclic_inventory=True)
    def a(asset,typ,key,**extra):return dict(base,asset_id=asset,asset_type=typ,technology=key,cost_key=key,**extra)
    rows=[a('electrolyzer_taranaki','electrolyzer','electrolyzer',efficiency=p['ely_eta'])]
    if case['power_twh']>0:
        rows.append(a('fuelcell_taranaki','h2_fuel_cell','h2_fuel_cell',efficiency=p['fc_eta']))
    if case['technology']=='Tank':
        rows.append(a('tank_taranaki','h2_store_tank','h2_tank'))
    else:
        rows += [a('uhs_injection_taranaki','uhs_injection','uhs_injection',efficiency=1),
                 a('uhs_withdrawal_taranaki','uhs_withdrawal','uhs_withdrawal',efficiency=1),
                 a('uhs_store_taranaki','uhs_store','uhs_storage')]
    return pd.DataFrame(rows)

def ptx_assets(case,p):
    is_nh3=case['ammonia_kt']>0
    is_meoh=case['methanol_kt']>0
    common=dict(zone='taranaki',available_from_year=2050,earliest_commissioning_year=2050,
                commissioning_type='flexible',enabled=True,installed_capacity_mw=0,
                installed_energy_tons=0,standing_loss=0)
    def a(name,kind,**kw):return dict(common,asset_id=name,asset_type=kind,technology=kind,cost_key=kind,**kw)
    rows=[]
    if is_nh3:
        nh3_mwh_per_t=p['hb_h2']*p['nh3_lhv']
        rows += [a('asu_taranaki','asu',electricity_mwh_per_t=p['asu_el']),
                 a('nh3_synthesis_taranaki','ammonia_synthesis',product='ammonia',
                   h2_mwh_per_t_product=nh3_mwh_per_t,
                   electricity_mwh_per_t=p['hb_el']*p['nh3_lhv'],
                   feedstock_t_per_t_product=p['hb_n2']*p['nh3_lhv']),
                 a('nh3_storage_taranaki','ammonia_store',product='ammonia')]
    if is_meoh:
        rows += [a('dac_taranaki','dac',electricity_mwh_per_t=p['dac_el']+p['dac_heat']/p['dac_COP']),
                 a('meoh_synthesis_taranaki','methanol_synthesis',product='methanol',
                   h2_mwh_per_t_product=p['meoh_h2']*p['meoh_lhv'],
                   electricity_mwh_per_t=p['meoh_el']*p['meoh_lhv'],
                   feedstock_t_per_t_product=p['meoh_co2']),
                 a('meoh_storage_taranaki','methanol_store',product='methanol')]
    return pd.DataFrame(rows)

def product_shipment_schedule(annual_tons, snapshots, ship_hours=48):
    """One monthly 48-h ship-loading event; annual delivery is exact.

    This replaces a monthly *aggregate* export target that could be served
    continually, thereby endogenising the need for final-product storage.
    Shipment day/hour is an illustrative assumption, not observed port data.
    """
    out=np.zeros(len(snapshots),dtype=float)
    if annual_tons<=0:return out
    if ship_hours<=0 or ship_hours>72:raise ValueError('ship_hours must be 1..72')
    for month in range(1,13):
        start=pd.Timestamp(year=2050,month=month,day=15)
        idx=(snapshots>=start)&(snapshots<start+pd.Timedelta(hours=ship_hours))
        assert idx.sum()==ship_hours
        out[idx]=(annual_tons/12)/ship_hours
    assert abs(out.sum()-annual_tons)<1e-5
    return out

In [6]:
def adapt_network_for_paper(n,case,p,snapshots):
    """Correct service isolation, tank compression and synthesis auxiliary power."""
    zone='taranaki'; el=f'elec_{zone}'; h2=f'h2_{zone}'
    if case['power_twh']>0:
        bus='electricity_output_taranaki'
        n.add('Bus',bus,carrier='electricity')
        n.links.loc['fuelcell_taranaki','bus1']=bus
        n.add('Load','ptH2tP_delivery',bus=bus,p_set=electricity_service(p,case,snapshots))
        # Output-only diagnostic slack: never count this as delivered PtH2tP energy.
        n.add('Generator','power_unserved_slack',bus=bus,carrier='slack',p_nom=1e7,
              marginal_cost=1e6)
    if case['technology']=='Tank':
        b='h2_tank_internal_taranaki'
        n.add('Bus',b,carrier='hydrogen_storage')
        n.stores.loc['tank_taranaki','bus']=b
        comp_capital=float(_read_required_csv('costs_2050_tank.csv').set_index('cost_key').loc['tank_injection','capital_cost'])
        n.add('Link','tank_injection_taranaki',bus0=h2,bus1=b,bus2=el,efficiency=1,
              efficiency2=-p['comp_electric'],carrier='h2_compression',p_nom_extendable=True,
              capital_cost=comp_capital)
        n.add('Link','tank_withdrawal_taranaki',bus0=b,bus1=h2,efficiency=1,
              carrier='h2_tank_withdrawal',p_nom_extendable=True,capital_cost=0)
    else:
        n.links.loc['uhs_injection_taranaki','bus2']=el
        n.links.loc['uhs_injection_taranaki','efficiency2']=-p['comp_electric']

    # Product demand as monthly ship-loading pulses instead of a continuous
    # export Link. This is essential for an informative PtX storage comparison.
    if case['ammonia_kt']>0:
        n.add('Load','ammonia_shipments',bus='nh3_taranaki',
              p_set=product_shipment_schedule(case['ammonia_kt']*1000,snapshots))
    if case['methanol_kt']>0:
        n.add('Load','methanol_shipments',bus='ch3oh_taranaki',
              p_set=product_shipment_schedule(case['methanol_kt']*1000,snapshots))

    for asset in ptx_assets(case,p).to_dict(orient='records'):
        if asset['asset_type'] not in ('ammonia_synthesis','methanol_synthesis'):continue
        aid=asset['asset_id']
        # Third optional bus of synthesis Link: direct electrical demand in MWh_e/t.
        n.links.loc[aid,'bus3']=el
        n.links.loc[aid,'efficiency3']=-asset['electricity_mwh_per_t']/asset['h2_mwh_per_t_product']
    return n

## 4. Network construction and optimisation

Keep the **original** `config.py`. This cell assembles the same PyPSA-Island network from CSVs using `build_case_network()`, adds only the PtX/UHS-specific wiring, and applies the optional annual H₂-throughput limit. The throughput limit represents **equivalent annual cycles**, not injection/withdrawal physics; site-specific DGR flow constraints remain a research limitation.

In [7]:
def _selected_case(case_id):
    match = scenarios_df.loc[scenarios_df["case_id"].eq(case_id)]
    if match.empty: raise KeyError(f"Unknown CASE_ID={case_id!r}. Check scenarios_2050.csv")
    return match.iloc[0].to_dict()

def _scale_wind_to_target(wind, target_cf):
    """Scale the same temporal shape to a target mean CF, clipping within [0,1]."""
    wind = np.asarray(wind, dtype=float)
    if target_cf is None: return wind
    target_cf = float(target_cf)
    if not 0 < target_cf < 1: raise ValueError("Target wind CF must lie strictly between 0 and 1")
    if (wind > 0).mean() + 1e-10 < target_cf:
        raise ValueError("Target CF cannot be obtained without changing the zero-resource hours")
    lo, hi = 0.0, 1e6
    for _ in range(65):
        mid = (lo + hi) / 2
        if np.minimum(wind * mid, 1).mean() < target_cf: lo = mid
        else: hi = mid
    return np.minimum(wind * hi, 1)


def _extra_constraints(n, selected_snapshots, case):
    config.add_custom_constraints(n, selected_snapshots)
    store_id = "tank_taranaki" if case["technology"] == "Tank" else "uhs_store_taranaki"
    withdrawal_id = "tank_withdrawal_taranaki" if case["technology"] == "Tank" else "uhs_withdrawal_taranaki"
    link_flow, _ = config._select_linopy_by_labels(
        n.model.variables["Link-p"], "Link", [withdrawal_id])
    storage_energy = config._select_single_linopy_label(
        n.model.variables["Store-e_nom"], "Store", store_id)
    n.model.add_constraints(
        link_flow.sum() <= float(case["max_fec"]) * storage_energy,
        name="annual_equivalent_H2_cycles",
    )


def build_pesgm_network(case_id, *, demo=False, target_wind_cf=None,
                        power_twh=None, ammonia_kt=None, methanol_kt=None):
    case = _selected_case(case_id)
    for field, value in [("power_twh", power_twh), ("ammonia_kt", ammonia_kt),
                         ("methanol_kt", methanol_kt)]:
        if value is not None:
            if value < 0: raise ValueError(f"{field} must be non-negative")
            case[field] = float(value)
    if demo:
        warnings.warn("SYNTHETIC renewable CF used for CODE TESTS ONLY — not paper results.", RuntimeWarning)
    wind, solar = make_hourly_profile(None if demo else PROFILE_CSV, demo=demo)
    wind = _scale_wind_to_target(wind, target_wind_cf)
    weather = {
        "wind": pd.DataFrame({"taranaki": wind}, index=snapshots),
        "solar": pd.DataFrame({"taranaki": solar}, index=snapshots),
    }
    nodes = config.load_nodes_from_csv(str(CASE_FOLDER / "nodes.csv"))
    generators = config.load_generator_capacity_from_csv(str(CASE_FOLDER / "generators_capacity.csv"))
    battery = config.load_storage_capacity_from_csv(str(CASE_FOLDER / "storage_capacity.csv"))
    n = config.build_case_network(
        snapshots=snapshots, nodes_df=nodes,
        load=pd.DataFrame({"taranaki": 0.0}, index=snapshots),
        interlinks=[], capacity_df=generators, storage_df=battery,
        hydro_assets_df=pd.DataFrame(),
        hydrogen_df=h2_assets(case, p), ptx_assets_df=ptx_assets(case, p),
        ptx_monthly_demand_df=pd.DataFrame(), vre_profiles=weather,
        year=MODEL_YEAR, enable_hydrogen=True, enable_ptx=True,
        general_settings={"slack_cost_per_mwh": 1e6},
        csv_costs=str(CASE_FOLDER / f"costs_2050_{case['technology'].lower()}.csv"),
    )
    adapt_network_for_paper(n, case, p, snapshots)
    config.sanitize_optional_link_ports(n)
    return n, case


def solve_pesgm_network(n, case):
    status, termination = n.optimize(
        solver_name=SOLVER,
        extra_functionality=lambda network, sn: _extra_constraints(network, sn, case),
    )
    if status != "ok" or termination not in ("optimal", "suboptimal"):
        raise RuntimeError(f"Optimisation failed: {status}, {termination}")
    if "power_unserved_slack" in n.generators.index:
        lost = float(n.generators_t.p["power_unserved_slack"].sum())
        if lost > 1e-3:
            raise RuntimeError(f"Unserved electricity = {lost:,.2f} MWh. Not a feasible PtH2tP design")
    return status, termination

## 5. Simulation, export and result loading

Simulations are independent and write their annual summaries and hourly data to disk. **Run only the selected cases**; results are reused unless overwrite is enabled. A saved result is tagged as synthetic when `DEMO_SYNTHETIC_CF=True`, so synthetic tests cannot silently enter manuscript figures.

In [8]:
def _optimised_capacity(frame, opt_column, fixed_column):
    optimized = pd.to_numeric(frame[opt_column], errors="coerce") if opt_column in frame else pd.Series(np.nan, index=frame.index)
    fixed = pd.to_numeric(frame[fixed_column], errors="coerce").fillna(0) if fixed_column in frame else pd.Series(0.0, index=frame.index)
    return optimized.fillna(fixed)

def _extract_case_results(n, case, *, synthetic=False):
    result = dict(case_id=case["case_id"], technology=case["technology"],
                  pathway=case["pathway"], dispatch=case["dispatch"],
                  power_twh=case["power_twh"], ammonia_kt=case["ammonia_kt"],
                  methanol_kt=case["methanol_kt"], synthetic=bool(synthetic),
                  annual_cost_eur=float(n.objective))
    generator_capacity = _optimised_capacity(n.generators, "p_nom_opt", "p_nom")
    link_capacity = _optimised_capacity(n.links, "p_nom_opt", "p_nom")
    store_capacity = _optimised_capacity(n.stores, "e_nom_opt", "e_nom")
    names = {
        "wind_mw": generator_capacity.get("wind_taranaki", 0.0),
        "solar_mw": generator_capacity.get("solar_taranaki", 0.0),
        "electrolyser_mw_in": link_capacity.get("electrolyzer_taranaki", 0.0),
        "fuel_cell_mw_in": link_capacity.get("fuelcell_taranaki", 0.0),
        "h2_storage_mwh": store_capacity.get("tank_taranaki", store_capacity.get("uhs_store_taranaki", 0.0)),
        "h2_injection_mw": link_capacity.get("tank_injection_taranaki", link_capacity.get("uhs_injection_taranaki", 0.0)),
        "h2_withdrawal_mw": link_capacity.get("tank_withdrawal_taranaki", link_capacity.get("uhs_withdrawal_taranaki", 0.0)),
        "nh3_storage_t": store_capacity.get("nh3_storage_taranaki", 0.0),
        "meoh_storage_t": store_capacity.get("meoh_storage_taranaki", 0.0),
    }
    result.update({k:float(v) for k,v in names.items()})
    result["fuel_cell_mw_out"] = result["fuel_cell_mw_in"]*p["fc_eta"]
    result["lcoe_eur_mwh"] = result["annual_cost_eur"]/(case["power_twh"]*1e6) if case["power_twh"]>0 and case["ammonia_kt"]==0 and case["methanol_kt"]==0 else np.nan
    result["lcoa_eur_t"] = result["annual_cost_eur"]/(case["ammonia_kt"]*1000) if case["ammonia_kt"]>0 and case["power_twh"]==0 else np.nan
    result["lcom_eur_t"] = result["annual_cost_eur"]/(case["methanol_kt"]*1000) if case["methanol_kt"]>0 and case["power_twh"]==0 else np.nan
    return result

def _export_case(n, case, *, synthetic=False, stem=None):
    stem = stem or case["case_id"]
    annual = _extract_case_results(n, case, synthetic=synthetic)
    pd.DataFrame([annual]).to_csv(SIMULATION_EXPORT_DIR / f"{stem}_summary.csv", index=False)
    if SAVE_NETWORK_NETCDF:
        n.export_to_netcdf(str(SIMULATION_EXPORT_DIR / f"{stem}_network.nc"))
    for label, timeseries in [
        ("generators", n.generators_t.p),
        ("links_p0", n.links_t.p0),
        ("stores_energy", n.stores_t.e),
        ("loads", n.loads_t.p),
    ]:
        timeseries.to_csv(SIMULATION_EXPORT_DIR / f"{stem}_{label}.csv")
    return annual


def _load_all_summaries():
    files = sorted(SIMULATION_EXPORT_DIR.glob("*_summary.csv"))
    if not files: return pd.DataFrame()
    results = pd.concat([pd.read_csv(path) for path in files], ignore_index=True)
    if "synthetic" in results.columns:
        results["synthetic"] = results["synthetic"].astype(str).str.lower().eq("true")
    return results

if RUN_SIMULATIONS:
    for _case_id in RUN_SCENARIO_CODES:
        _file = SIMULATION_EXPORT_DIR / f"{_case_id}_summary.csv"
        if _file.exists() and not OVERWRITE_EXISTING_EXPORTS:
            _cached = pd.read_csv(_file)
            if _cached.get("synthetic", pd.Series([False])).astype(str).str.lower().eq("true").any() and not DEMO_SYNTHETIC_CF:
                raise RuntimeError(f"Cached {_case_id} uses synthetic weather. Set OVERWRITE_EXISTING_EXPORTS=True to replace it with real-weather results.")
            print(f"Skipping {_case_id}: cached output exists (OVERWRITE_EXISTING_EXPORTS=False)")
            continue
        print(f"Optimising {_case_id}...")
        _n, _case = build_pesgm_network(_case_id, demo=DEMO_SYNTHETIC_CF)
        _status, _termination = solve_pesgm_network(_n, _case)
        _record = _export_case(_n, _case, synthetic=DEMO_SYNTHETIC_CF)
        print(f"  {_status}/{_termination}: annual cost €{_record['annual_cost_eur']/1e6:,.1f}m")
        del _n
        gc.collect()
else:
    print("RUN_SIMULATIONS=False; no optimisation executed. Existing exports can still be plotted.")

results_all = _load_all_summaries()
# Exclude synthetic test results from paper figures.
results_paper = results_all.loc[~results_all["synthetic"]].copy() if not results_all.empty else pd.DataFrame()
if not results_all.empty:
    display(results_all[["case_id", "technology", "pathway", "dispatch", "annual_cost_eur", "synthetic"]])

RUN_SIMULATIONS=False; no optimisation executed. Existing exports can still be plotted.


## 6. Main-paper figures

All figures share the Punta Arenas palette, font sizes, two-column width, axis helpers, panel labels, automatic layout and **300-dpi PNG + editable vector SVG** export. Figures are based **only on completed, non-synthetic PyPSA optimisations**. Each plotting cell prints a helpful message instead of failing when there are no solved cases.

### Figure 2 — System cost and technology comparison

Compare **annual system cost for identical pathway/demand/dispatch configurations**. Absolute costs are used for mixed electricity + PtX systems; a single LCOX would arbitrarily allocate common costs. The figure keeps distinct storage colours and reports direct savings relative to Tank only when an exact comparator exists.

In [9]:
def _service_key(row):
    return (str(row["pathway"]), str(row["dispatch"]) if float(row["power_twh"]) > 0 else "none",
            float(row["power_twh"]), float(row["ammonia_kt"]), float(row["methanol_kt"]))

if results_paper.empty:
    print("Figure 2: no real-weather optimisations yet. Enable RUN_SIMULATIONS after adding Taranaki_CF_2050.csv")
else:
    ordered = results_paper.sort_values(["pathway", "dispatch", "technology"]).copy()
    ordered["scenario_label"] = [
        PATHWAY_LABELS.get(r.pathway, r.pathway) + (" · " + r.dispatch if r.dispatch != "none" else "")
        for r in ordered.itertuples(index=False)
    ]
    fig, ax = plt.subplots(figsize=(FIGSIZE_2COL_LARGE[0], max(FIGSIZE_2COL_LARGE[1], 0.31*len(ordered)+1.3)))
    y = np.arange(len(ordered))
    colors = ordered["technology"].map(STORAGE_COLORS)
    ax.barh(y, ordered["annual_cost_eur"]/1e6, color=colors, edgecolor="white", linewidth=.4)
    ax.set_yticks(y, labels=(ordered["case_id"] + "  " + ordered["scenario_label"]).tolist())
    ax.invert_yaxis()
    _paper_axis(ax, xlabel="Annualised system cost [million EUR/yr]", grid_axis="x")
    _paper_panel_label(ax, "(a)")
    ax.set_xlim(left=0)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor=color, label=name) for name, color in STORAGE_COLORS.items()],
              loc="lower right", frameon=False, ncol=3)
    _save_paper_figure(fig, "Fig2_annual_system_cost")
    plt.show()
    # Matched comparisons, never extrapolate daily-vs-seasonal or mismatched volumes.
    comp = ordered.copy()
    comp["service_key"] = comp.apply(_service_key, axis=1)
    base = comp.loc[comp["technology"].eq("Tank"), ["service_key", "annual_cost_eur"]]
    base = base.drop_duplicates("service_key").rename(columns={"annual_cost_eur":"tank_cost_eur"})
    comp = comp.merge(base, on="service_key", how="left")
    comp["saving_vs_tank_pct"] = 100*(1-comp["annual_cost_eur"]/comp["tank_cost_eur"])
    comp[["case_id","technology","pathway","dispatch","annual_cost_eur","tank_cost_eur","saving_vs_tank_pct"]].to_csv(
        TABLE_EXPORT_DIR / "storage_cost_comparisons.csv", index=False)
    display(comp[["case_id", "annual_cost_eur", "saving_vs_tank_pct"]])

Figure 2: no real-weather optimisations yet. Enable RUN_SIMULATIONS after adding Taranaki_CF_2050.csv


### Figure 3 — Optimised power and energy capacities

Two complementary panels: renewable/conversion **power** (GW) and hydrogen storage **working energy** (GWh). H₂ stock and throughput are not conflated with well power. For SC/DGR injection and withdrawal capacity see the exported capacity table.

In [10]:
if results_paper.empty:
    print("Figure 3: no real-weather capacity optimisation results yet.")
else:
    fig, axes = plt.subplots(2, 1, figsize=(FIGSIZE_2COL_LARGE[0], max(FIGSIZE_PAPER_2X2[1], 5.6)), sharex=True)
    df = results_paper.sort_values(["pathway", "dispatch", "technology"])
    xpos = np.arange(len(df))
    for key, label, color in [
        ("wind_mw", "Wind", TECH_COLORS["wind"]),
        ("solar_mw", "Solar PV", TECH_COLORS["solar"]),
        ("electrolyser_mw_in", "Electrolyser", TECH_COLORS["electrolyzer"]),
        ("fuel_cell_mw_out", "Fuel cell (output)", TECH_COLORS["h2_fuel_cell"]),
    ]:
        # Grouped marks rather than stacking input and output capacity as energy.
        offset = ["wind_mw","solar_mw","electrolyser_mw_in","fuel_cell_mw_out"].index(key)
        axes[0].bar(xpos - .30 + .2*offset, df[key].to_numpy()/1000, width=.19,
                    color=color, label=label)
    _paper_axis(axes[0], ylabel="Installed power [GW]")
    _paper_panel_label(axes[0], "(a)")
    axes[0].legend(loc="upper right", frameon=False, ncol=2)
    axes[1].bar(xpos, df["h2_storage_mwh"].to_numpy()/1000,
                color=df["technology"].map(STORAGE_COLORS), width=.72)
    _paper_axis(axes[1], ylabel="H₂ working capacity [GWh]", grid_axis="y")
    _paper_panel_label(axes[1], "(b)")
    axes[1].set_xticks(xpos, df["case_id"].tolist(), rotation=45, ha="right")
    _save_paper_figure(fig, "Fig3_optimised_power_and_storage")
    plt.show()
    df[["case_id", "technology", "power_twh", "ammonia_kt", "methanol_kt", "wind_mw", "solar_mw",
        "electrolyser_mw_in", "fuel_cell_mw_out", "h2_storage_mwh", "h2_injection_mw",
        "h2_withdrawal_mw", "nh3_storage_t", "meoh_storage_t"]].to_csv(
        TABLE_EXPORT_DIR / "optimised_capacities.csv", index=False)

Figure 3: no real-weather capacity optimisation results yet.


### Figure 4 — Storage inventory and hourly dispatch (active case)

Annual H₂ stock shows seasonal cycling; the weekly zoom contrasts renewable generation, electrolysis and fuel-cell output. The figure reads exported hourly CSVs, so it works even when the solver is not rerun.

In [11]:
def _hourly_result(case_id, suffix):
    file = SIMULATION_EXPORT_DIR / f"{case_id}_{suffix}.csv"
    if not file.exists(): raise FileNotFoundError(file)
    return pd.read_csv(file, index_col=0, parse_dates=True)

_active_summary = results_paper.loc[results_paper["case_id"].eq(ACTIVE_SCENARIO)] if not results_paper.empty else pd.DataFrame()
if _active_summary.empty:
    print(f"Figure 4: optimise {ACTIVE_SCENARIO} with real profiles to create its hourly output.")
else:
    inventory = _hourly_result(ACTIVE_SCENARIO, "stores_energy")
    generation = _hourly_result(ACTIVE_SCENARIO, "generators")
    link_dispatch = _hourly_result(ACTIVE_SCENARIO, "links_p0")
    store_name = "tank_taranaki" if "tank_taranaki" in inventory else "uhs_store_taranaki"
    if store_name not in inventory:
        raise ValueError(f"No H₂ store timeseries for {ACTIVE_SCENARIO}")
    fig, axes = plt.subplots(2,1,figsize=FIGSIZE_PAPER_2X2)
    axes[0].plot(inventory.index, inventory[store_name]/1000,
                 color=STORAGE_COLORS[str(_active_summary.iloc[0]["technology"])], lw=1.35)
    _paper_axis(axes[0], ylabel="H₂ inventory [GWh]")
    _paper_panel_label(axes[0], "(a)")
    axes[0].set_title(f"Storage inventory — {ACTIVE_SCENARIO}", fontsize=FONT_SIZE_LABEL)
    start = pd.Timestamp(f"{MODEL_YEAR}-07-01")
    finish = start + pd.Timedelta(days=7)
    hours = (generation.index >= start) & (generation.index < finish)
    if "wind_taranaki" in generation:
        axes[1].plot(generation.index[hours],generation.loc[hours,"wind_taranaki"],
                     color=TECH_COLORS["wind"],label="Wind",lw=1.1)
    if "solar_taranaki" in generation:
        axes[1].plot(generation.index[hours],generation.loc[hours,"solar_taranaki"],
                     color=TECH_COLORS["solar"],label="Solar",lw=1.1)
    if "electrolyzer_taranaki" in link_dispatch:
        axes[1].plot(link_dispatch.index[hours],link_dispatch.loc[hours,"electrolyzer_taranaki"],
                     color=TECH_COLORS["electrolyzer"],label="Electrolyser electricity",lw=1.2)
    if "fuelcell_taranaki" in link_dispatch:
        axes[1].plot(link_dispatch.index[hours],link_dispatch.loc[hours,"fuelcell_taranaki"]*p["fc_eta"],
                     color=TECH_COLORS["h2_fuel_cell"],label="Fuel cell output",lw=1.2)
    _paper_axis(axes[1], ylabel="Power [MW]", xlabel="July 2050")
    _paper_panel_label(axes[1], "(b)")
    axes[1].legend(loc="upper right",frameon=False,ncol=2)
    _save_paper_figure(fig, f"Fig4_H2_inventory_dispatch_{ACTIVE_SCENARIO}")
    plt.show()

Figure 4: optimise C-S2 with real profiles to create its hourly output.


## 7. Supplementary analysis — production scale × renewable capacity factor

The sensitivity **re-optimises the full hourly system**, unlike the Excel screening. One axis scales annual demand (same annual product/service), and the other scales the *same hourly wind shape* to a selected mean CF. It reports **percentage savings relative to a Tank case with identical service and profiles**. This is off by default because each grid point is a full optimisation.

For this first validation, start with PtA: Tank (`T-S2`) vs SC (`C-S2`) or DGR (`D-S2`). Once the results are reliable, expand to PtM and shared pathways. The CF transformation alone does not capture changes in renewable variability or spatial composition; those require additional weather profiles.

In [12]:
# ── Sensitivity controls ───────────────────────────────────────────────────
SENSITIVITY_PATHWAY = "S2"             # S1, S2 or S3 for the first comparison
SENSITIVITY_TECHNOLOGIES = ["Tank", "SC"]  # Add DGR after flow calibration
PRODUCTION_MULTIPLIERS = [0.5, 1.0, 2.0]
TARGET_WIND_CFS = [0.35, 0.45, 0.55]
SENSITIVITY_STORAGE_CODES = {"Tank": "T", "DGR": "D", "SC": "C"}

if RUN_SENSITIVITY:
    if DEMO_SYNTHETIC_CF:
        raise RuntimeError("Do not run a publishable sensitivity with synthetic resource data")
    if SENSITIVITY_PATHWAY not in {"S1", "S2", "S3"}:
        raise ValueError("The first sensitivity supports single-service S1/S2/S3 only")
    sweep_rows = []
    for tech in SENSITIVITY_TECHNOLOGIES:
        prefix = SENSITIVITY_STORAGE_CODES[tech]
        case_id = f"{prefix}-{SENSITIVITY_PATHWAY}" + ("-S" if SENSITIVITY_PATHWAY=="S1" else "")
        reference = _selected_case(case_id)
        for mult in PRODUCTION_MULTIPLIERS:
            for target_cf in TARGET_WIND_CFS:
                kwargs = {key: float(reference[key])*mult for key in ["power_twh", "ammonia_kt", "methanol_kt"]}
                print(f"Solving {case_id}: demand ×{mult}, mean wind CF={target_cf:.0%}")
                model, case = build_pesgm_network(case_id, target_wind_cf=target_cf, **kwargs)
                solve_pesgm_network(model, case)
                res = _extract_case_results(model, case, synthetic=False)
                res.update(technology=tech, multiplier=mult, wind_cf_target=target_cf)
                sweep_rows.append(res)
                del model
                gc.collect()
                pd.DataFrame(sweep_rows).to_csv(TABLE_EXPORT_DIR/"hourly_scale_CF_sensitivity.csv",index=False)
else:
    print("RUN_SENSITIVITY=False: the hourly scale × CF sweep is disabled.")

_sensitivity_csv = TABLE_EXPORT_DIR/"hourly_scale_CF_sensitivity.csv"
if _sensitivity_csv.exists():
    sweep = pd.read_csv(_sensitivity_csv)
    if {"technology","multiplier","wind_cf_target","annual_cost_eur"}.issubset(sweep.columns):
        tank = sweep[sweep["technology"]=="Tank"][["multiplier","wind_cf_target","annual_cost_eur"]].rename(columns={"annual_cost_eur":"tank_eur"})
        savings = sweep.merge(tank,on=["multiplier","wind_cf_target"],how="inner")
        savings["saving_pct"] = 100*(1-savings["annual_cost_eur"]/savings["tank_eur"])
        non_tank = savings[savings["technology"]!="Tank"]
        if not non_tank.empty:
            tech = non_tank["technology"].iloc[0]
            z = non_tank[non_tank["technology"]==tech].pivot(index="multiplier",columns="wind_cf_target",values="saving_pct")
            if z.notna().all().all():
                fig, ax = plt.subplots(figsize=FIGSIZE_2COL)
                im = ax.imshow(z,origin="lower",aspect="auto",cmap="RdBu",vmin=-abs(z).max().max(),vmax=abs(z).max().max())
                ax.set_xticks(np.arange(len(z.columns)),[f"{v:.0%}" for v in z.columns])
                ax.set_yticks(np.arange(len(z.index)),[f"×{v:g}" for v in z.index])
                _paper_axis(ax,xlabel="Mean wind capacity factor",ylabel="Annual production multiplier",grid_axis=None)
                _paper_panel_label(ax,"(a)")
                fig.colorbar(im,ax=ax,label=f"{tech} saving vs Tank [%]")
                _save_paper_figure(fig,f"Fig5_scale_CF_{tech}_vs_Tank")
                plt.show()
        display(savings[["technology","multiplier","wind_cf_target","saving_pct"]])
else:
    print("No hourly sensitivity table saved yet.")

RUN_SENSITIVITY=False: the hourly scale × CF sweep is disabled.
No hourly sensitivity table saved yet.


## 8. Diagnostics, limitations and recommended execution order

**First run:** create a real `input_uhs_pesgm/Taranaki_CF_2050.csv` with exactly **8,760 hourly rows**, ordered January–December 2050, containing `wind_cf,solar_cf` values in `[0,1]`. Leave `DEMO_SYNTHETIC_CF=False`. Start with `RUN_SCENARIO_CODES=["T-S2","C-S2"]` and `RUN_SIMULATIONS=True`. Compare their balances and capacities before running DGR or combined cases. Then enable sensitivity sweeps only after validating single-case results.

**Scientific checks before interpreting comparisons:**

- The SC comparison is hypothetical for Taranaki; geological availability is not established.
- DGR well power and productivity are not calibrated; screening CAPEX / site size remain provisional.
- The 48-hour monthly shipment pulses and the seasonal electricity service shape are explicit assumptions.
- Annualised costs are already stored in the technology-cost CSVs. Never annualise them a second time.
- Verify PyPSA hourly bus balances, load delivery, inventory trajectories and the H₂ cycle constraint against the chosen weather year.
- Compare annual system cost only for equal demand and time profiles; for combined PtX routes do not report a falsely allocated single-product LCOX.
- The original `config.py` imports `build_demand.py` even though the direct network builder does not require it; an **in-memory import placeholder** is used only if the file is unavailable. `config.py` is not modified.

**Notebook structure and plots** intentionally match `case_analysis_PuntaArenas.ipynb` in section order, figure styles, colour codes, saved file formats and diagnostic outputs.